# AWS Bedrock + LiteLLM Testing

Simple tests for Bedrock models via LiteLLM with cost tracking.

In [1]:
# Imports
import os

import dspy
from datasets import load_dataset
from dotenv import load_dotenv
from dspy import GEPA, LM
from litellm import completion, completion_cost

load_dotenv()
print("✓ Loaded")

✓ Loaded


In [2]:
# Config
region = os.getenv("AWS_REGION")
token = os.getenv("AWS_BEARER_TOKEN_BEDROCK")

llama_17b = os.getenv("BEDROCK_LLAMA_17B")
llama_70b = os.getenv("BEDROCK_LLAMA_70B")
llama_8b = os.getenv("BEDROCK_LLAMA_8B")
gpt_oss_20b = os.getenv("BEDROCK_GPT_OSS_20B")
gpt_oss_120b = os.getenv("BEDROCK_GPT_OSS_120B")

print(f"Region: {region}")
print(f"Token: {'✓' if token else '✗'}")
print(f"Llama Models: {llama_17b}, {llama_70b}, {llama_8b}")
print(f"GPT OSS Models: {gpt_oss_20b}, {gpt_oss_120b}")

Region: us-east-1
Token: ✓
Llama Models: us.meta.llama4-scout-17b-instruct-v1:0, us.meta.llama3-1-70b-instruct-v1:0, meta.llama3-8b-instruct-v1:0
GPT OSS Models: amazon.nova-micro-v1:0, amazon.nova-pro-v1:0


## Test 1: Basic Call

In [3]:
response = completion(
    model=f"bedrock/{llama_8b}",
    messages=[{"role": "user", "content": "Say hello in 5 words"}],
    max_tokens=50,
    temperature=0,
    api_key=token,
)

print(f"Response: {response.choices[0].message.content}")
print(f"Tokens: {response.usage.total_tokens}")
print(f"Cost: ${completion_cost(completion_response=response):.6f}")

Response: 

Hello, how are you today?
Tokens: 29
Cost: $0.000011


## Test 1b: GPT OSS 120B

In [4]:
response = completion(
    model=f"bedrock/{gpt_oss_120b}",
    messages=[{"role": "user", "content": "Say hello in 5 words"}],
    max_tokens=200,
    temperature=0,
)

print(f"Response: {response.choices[0].message.content}")
print(f"Tokens: {response.usage.total_tokens}")
print(f"Cost: ${completion_cost(completion_response=response):.6f}")

Response: Hello, it's great to connect!
Tokens: 16
Cost: $0.000037


## Test 2: Math Reasoning

In [5]:
response = completion(
    model=f"bedrock/{llama_8b}",
    messages=[
        {"role": "system", "content": "Think step by step"},
        {
            "role": "user",
            "content": "If I have 3 apples and buy 2 more, then give 1 away, how many?",
        },
    ],
    max_tokens=150,
    temperature=0,
)

print(response.choices[0].message.content)
print(f"\nTokens: {response.usage.total_tokens}")
print(f"Cost: ${completion_cost(completion_response=response):.6f}")



Let's break it down step by step:

1. You start with 3 apples.
2. You buy 2 more apples, so now you have:
3 (initial apples) + 2 (new apples) = 5 apples
3. You give 1 apple away, so you subtract 1 from the total:
5 apples - 1 apple = 4 apples

So, you have 4 apples left.

Tokens: 128
Cost: $0.000065


## Test 3: Compare All Models

In [6]:
question = "What is 2+2?"
total_cost = 0

for name, model in [("17B", llama_17b), ("70B", llama_70b), ("8B", llama_8b)]:
    response = completion(
        model=f"bedrock/{model}",
        messages=[{"role": "user", "content": question}],
        max_tokens=20,
        temperature=0,
    )

    cost = completion_cost(completion_response=response)
    total_cost += cost

    print(f"{name}: {response.choices[0].message.content}")
    print(f"     Tokens: {response.usage.total_tokens}, Cost: ${cost:.6f}\n")

print(f"Total: ${total_cost:.6f}")

17B: 2 + 2 = 4.
     Tokens: 51, Cost: $0.000013

70B: 

2 + 2 = 4
     Tokens: 30, Cost: $0.000030

8B: 

The answer to 2+2 is 4.
     Tokens: 34, Cost: $0.000014

Total: $0.000057


## Test 5: Temperature Effects

In [7]:
prompt = "The future of AI is"

for temp in [0.0, 0.7, 1.0]:
    response = completion(
        model=f"bedrock/{llama_8b}",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=30,
        temperature=temp,
    )

    print(f"Temp {temp}: {response.choices[0].message.content}\n")

Temp 0.0: 

The future of AI is likely to be shaped by various factors, including advancements in technology, changes in societal needs, and the development of new applications

Temp 0.7: 

A topic that sparks much excitement and curiosity! Here are some potential developments that could shape the future of AI:

1. **Advancements in Natural

Temp 1.0: 

A topic that sparks much curiosity and excitement! The future of AI is expected to be shaped by various factors, including advancements in technology, changing societal



## Test 6: DSPy with LiteLLM

In [8]:
# Configure DSPy to use LiteLLM with Bedrock
lm = LM(model=f"bedrock/{llama_8b}", temperature=0, max_tokens=100)
dspy.configure(lm=lm)


# Define a simple signature
class BasicQA(dspy.Signature):
    """Answer questions concisely."""

    question: str = dspy.InputField()
    answer: str = dspy.OutputField()


# Create and test predictor
qa = dspy.ChainOfThought(BasicQA)
question = "What is the capital of France?"
response = qa(question=question)

print(f"Question: {question}")
print(f"Answer: {response.answer}")

# Inspect available fields
print(f"\nAvailable fields: {list(response.keys())}")

# Try to access reasoning if available
if hasattr(response, "reasoning"):
    print(f"\nReasoning: {response.reasoning}")
elif "reasoning" in response:
    print(f"\nReasoning: {response.reasoning}")

Question: What is the capital of France?
Answer: Paris

Available fields: ['reasoning', 'answer']

Reasoning: The capital of France is the largest city in the country and is known for its iconic landmarks such as the Eiffel Tower and Notre-Dame Cathedral.


In [9]:
# Access DSPy call history for cost tracking

# Get the last call from DSPy's history
last_call = lm.history[-1]

print("DSPy Call History:")
print(f"  Prompt tokens: {last_call['usage']['prompt_tokens']}")
print(f"  Completion tokens: {last_call['usage']['completion_tokens']}")
print(f"  Total tokens: {last_call['usage']['total_tokens']}")

# Calculate cost from the response
if "response" in last_call:
    cost = completion_cost(completion_response=last_call["response"])
    print(f"  Cost: ${cost:.6f}")
else:
    print("\n  Note: Raw response not available in history")
    print("  You can enable this with: lm = LM(..., cache=False)")

DSPy Call History:
  Prompt tokens: 168
  Completion tokens: 50
  Total tokens: 218
  Cost: $0.000080


# GEPA/DSPy HotpotQA Task

Testing prompt optimization on multi-hop question answering.

In [10]:
def init_dataset(num_samples=100):
    # Load HotpotQA dataset
    raw_data = load_dataset("hotpot_qa", "fullwiki")

    # Process train split
    raw_train = (
        raw_data["train"].shuffle(seed=42).select(range(min(len(raw_data["train"]), num_samples)))
    )
    train_split = [
        dspy.Example(
            {
                "question": x["question"],
                "answer": x["answer"],
            }
        ).with_inputs("question")
        for x in raw_train
    ]

    # Process validation split for test
    raw_val = (
        raw_data["validation"]
        .shuffle(seed=42)
        .select(range(min(len(raw_data["validation"]), num_samples)))
    )
    test_split = [
        dspy.Example(
            {
                "question": x["question"],
                "answer": x["answer"],
            }
        ).with_inputs("question")
        for x in raw_val
    ]

    # Split train into train/val
    tot_num = len(train_split)
    train_set = train_split[: int(0.5 * tot_num)]
    val_set = train_split[int(0.5 * tot_num) :]
    test_set = test_split

    return train_set, val_set, test_set


train_set, val_set, test_set = init_dataset(num_samples=100)
print(f"Train: {len(train_set)}, Val: {len(val_set)}, Test: {len(test_set)}")
print(f"Example: {train_set[0]}")

Train: 50, Val: 50, Test: 100
Example: Example({'question': 'Which airport is located in Maine, Sacramento International Airport or Knox County Regional Airport?', 'answer': 'Knox County Regional Airport'}) (input_keys={'question'})


In [11]:
# Configure DSPy to use LiteLLM with Bedrock (task agent)
lm = LM(model=f"bedrock/{gpt_oss_20b}", temperature=0.7, max_tokens=8192)
dspy.configure(lm=lm)

In [12]:
class GenerateResponse(dspy.Signature):
    """Answer the question with a short, factual response."""

    question = dspy.InputField()
    answer = dspy.OutputField(desc="A short factual answer (1-5 words)")


program = dspy.ChainOfThought(GenerateResponse)


def normalize_answer(s):
    """Normalize answer for comparison."""
    if s is None:
        return ""
    return s.lower().strip()


def metric(example, prediction, trace=None, pred_name=None, pred_trace=None):
    gold = normalize_answer(example["answer"])
    pred = normalize_answer(prediction.answer)
    if not pred:
        return 0
    # Exact match or gold contained in prediction
    return int(gold == pred or gold in pred or pred in gold)

In [13]:
evaluate = dspy.Evaluate(
    devset=test_set,
    metric=metric,
    num_threads=32,
    display_table=True,
    display_progress=True,
)

evaluate(program)

Average Metric: 26.00 / 100 (26.0%): 100%|██████████| 100/100 [00:02<00:00, 36.67it/s]

2026/04/19 17:04:14 INFO dspy.evaluate.evaluate: Average Metric: 26 / 100 (26.0%)


,question,example_answer,reasoning,pred_answer,metric
0,What nationality was Oliver Reed's character in the film Royal Flash?,Prussian,"Oliver Reed's character in the film ""Royal Flash"" is known as King...",British,✔️ [0]
1,Pacific Mozart Ensemble performed which German composer's Der Lind...,Kurt Julian Weill,"The Pacific Mozart Ensemble performed the work ""Der Lindberghflug""...",Kurt Weill,✔️ [0]
2,"Who released the song ""With or Without You"" first, Jai McDowall or...",U2,"The song ""With or Without You"" was originally released by Jai McDo...",Jai McDowall,✔️ [0]
3,"What Kentucky county has a population of 60,316 and features the L...",Oldham County,"The Lake Louisvilla neighborhood is part of Jefferson County, Kent...",Jefferson County,✔️ [0]
4,"Para Hills West, South Australia lies within a city with what esti...","138,535","Para Hills West is a suburb located in the city of Adelaide, South...",1.3 million,✔️ [0]
...,...,...,...,...,...
95,"What company has a headquarters in Whitley, Convernty, United King...",Jaguar Land Rover,"The company with headquarters in Whitley, Coventry, United Kingdom...",Jaguar Land Rover,✔️ [1]
96,Who starred as an American attorney in Grey Gardens?,Ken Howard,"The film ""Grey Gardens"" is a documentary that features the eccentr...",Jackie and Edie Beale,✔️ [0]
97,Was Princess Charlotte of Cambridge born before or after the repea...,after,The Royal Marriages Act 1772 required the consent of the monarch f...,after repeal,✔️ [1]
98,Hermann Wilhelm Göring was a veteran fighter pilot in a war that e...,1918,Hermann Wilhelm Göring was a German military aviator and politicia...,1918,✔️ [1]


EvaluationResult(score=26.0, results=<list of 100 results>)

In [14]:
def metric_with_feedback(example, prediction, trace=None, pred_name=None, pred_trace=None):
    gold = normalize_answer(example["answer"])
    pred = normalize_answer(prediction.answer)

    # Handle None/empty answer
    if not pred:
        feedback_text = (
            "Your response was empty or could not be parsed. "
            "Please provide a short, factual answer to the question."
        )
        feedback_text += f" The correct answer is '{example['answer']}'."
        return dspy.Prediction(score=0, feedback=feedback_text)

    # Check for match
    score = int(gold == pred or gold in pred or pred in gold)

    if score == 1:
        feedback_text = f"Correct! The answer is '{example['answer']}'."
    else:
        feedback_text = f"Incorrect. You answered '{prediction.answer}', but the correct answer is '{example['answer']}'."
        feedback_text += " Make sure to provide a concise, factual answer."

    return dspy.Prediction(score=score, feedback=feedback_text)

In [15]:
optimizer = GEPA(
    metric=metric_with_feedback,
    auto="light",
    num_threads=64,
    track_stats=True,
    reflection_minibatch_size=3,
    # Using gpt_oss_120b for reflection - larger model for instruction generation
    reflection_lm=dspy.LM(model=f"bedrock/{gpt_oss_120b}", temperature=1.0, max_tokens=4096),
)

optimized_program = optimizer.compile(
    program,
    trainset=train_set,
    valset=val_set,
)

2026/04/19 17:04:14 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 580 metric calls of the program. This amounts to 5.80 full evals on the train+val set.
2026/04/19 17:04:14 INFO dspy.teleprompt.gepa.gepa: Using 50 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget. GEPA requires you to provide the smallest valset that is just large enough to match your downstream task distribution, while providing as large trainset as possible.
GEPA Optimization:   0%|          | 0/580 [00:00<?, ?rollouts/s]2026/04/19 17:04:15 INFO dspy.evaluate.evaluate: Average Metric: 11.0 / 50 (22.0%)
2026/04/19 17:04:15 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.22 over 50 / 50 examples
GEPA Optimization:   9%|▊         | 50/580 [00:01<00:11, 48.00rollouts/s]2026/04/19 17:04:15 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Selected program 0 score: 0.22


Average Metric: 1.00 / 3 (33.3%): 100%|██████████| 3/3 [00:00<00:00,  4.70it/s]

2026/04/19 17:04:16 INFO dspy.evaluate.evaluate: Average Metric: 1.0 / 3 (33.3%)


2026/04/19 17:04:18 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Proposed new text for predict: Your task is to provide a short, factual response to the given question. Ensure the answer is concise and accurate. 

When addressing the question, consider the following guidelines:
1. **Geographical Comparisons**: For questions comparing locations, such as which place is farther west, ensure you accurately determine the relative positions based on reliable geographical data.
   - For instance, when comparing Tehran, Iran, and Riyadh, Saudi Arabia, Tehran is actually farther west than Riyadh.

2. **Occupations and Historical Figures**: When asked about the mutual occupation of historical figures, provide the correct profession. Verify the information from credible sources to confirm their contributions and roles.
   - Example: Al-Battani and Ibn al-Shatir were both astronomers.

3. **Character Information from Media**: For questions about characters from films or TV, provide specific factua

Average Metric: 0.00 / 3 (0.0%): 100%|██████████| 3/3 [00:00<00:00,  4.24it/s]

...

2026/04/19 17:06:23 INFO dspy.evaluate.evaluate: Average Metric: 2.0 / 3 (66.7%)


2026/04/19 17:06:25 INFO dspy.teleprompt.gepa.gepa: Iteration 10: Proposed new text for predict: New Instruction:

When answering questions, provide a short, factual response that directly addresses the query. Ensure that the information is accurate and concise. Here are some guidelines to follow:

1. **Identify the Key Query**: Determine the main question being asked. Focus on the specific information requested.

2. **Provide a Direct Answer**: Answer the question without additional commentary or explanation. Keep the response brief and to the point.

3. **Ensure Accuracy**: Verify the factual correctness of your answer. If the question pertains to a specific domain (e.g., sports, TV shows, literature), make sure you have the correct and up-to-date information.

4. **Avoid Assumptions**: Do not make assumptions beyond what is asked in the question. Stick to the facts presented.

5. **Use Proper Formatting**: Present the answer in a clear and readable format.

6. **Domain-Specific Know

Average Metric: 2.00 / 3 (66.7%): 100%|██████████| 3/3 [00:00<00:00,  5.40it/s] 

2026/04/19 17:06:26 INFO dspy.evaluate.evaluate: Average Metric: 2.0 / 3 (66.7%)


2026/04/19 17:06:29 INFO dspy.teleprompt.gepa.gepa: Iteration 11: Proposed new text for predict: Your task is to provide a short, factual response to the given question. Ensure the answer is concise and accurate. 

When addressing the question, consider the following guidelines:
1. **Geographical Comparisons**: For questions comparing locations, such as which place is farther west, ensure you accurately determine the relative positions based on reliable geographical data.
   - For instance, when comparing Tehran, Iran, and Riyadh, Saudi Arabia, Tehran is actually farther west than Riyadh.

2. **Occupations and Historical Figures**: When asked about the mutual occupation of historical figures, provide the correct profession. Verify the information from credible sources to confirm their contributions and roles.
   - Example: Al-Battani and Ibn al-Shatir were both astronomers.

3. **Character Information from Media**: For questions about characters from films or TV, provide specific factu

In [16]:
print(optimized_program.predict.signature.instructions)

Your task is to provide a short, factual response to the given question. Ensure the answer is concise and accurate. 

When addressing the question, consider the following guidelines:
1. **Geographical Comparisons**: For questions comparing locations, such as which place is farther west, ensure you accurately determine the relative positions based on reliable geographical data.
   - For instance, when comparing Tehran, Iran, and Riyadh, Saudi Arabia, Tehran is actually farther west than Riyadh.

2. **Occupations and Historical Figures**: When asked about the mutual occupation of historical figures, provide the correct profession. Verify the information from credible sources to confirm their contributions and roles.
   - Example: Al-Battani and Ibn al-Shatir were both astronomers.

3. **Character Information from Media**: For questions about characters from films or TV, provide specific factual details if they are available and verified. If the information is not available or verifiable,

In [18]:
evaluate(optimized_program)

Average Metric: 30.00 / 100 (30.0%): 100%|██████████| 100/100 [00:01<00:00, 64.06it/s]

2026/04/19 17:12:02 INFO dspy.evaluate.evaluate: Average Metric: 30 / 100 (30.0%)


,question,example_answer,reasoning,pred_answer,metric
0,What nationality was Oliver Reed's character in the film Royal Flash?,Prussian,Oliver Reed played the character Harry Paget Flashman in the film ...,English,✔️ [0]
1,Pacific Mozart Ensemble performed which German composer's Der Lind...,Kurt Julian Weill,"The Pacific Mozart Ensemble performed Carl Orff's ""Der Lindberghfl...",Carl Orff,✔️ [0]
2,"Who released the song ""With or Without You"" first, Jai McDowall or...",U2,"To determine who released the song ""With or Without You"" first, we...",U2,✔️ [1]
3,"What Kentucky county has a population of 60,316 and features the L...",Oldham County,The question is asking for a specific Kentucky county that has a p...,Jefferson County,✔️ [0]
4,"Para Hills West, South Australia lies within a city with what esti...","138,535",Para Hills West is a suburb located within the City of Playford in...,"118,810",✔️ [0]
...,...,...,...,...,...
95,"What company has a headquarters in Whitley, Convernty, United King...",Jaguar Land Rover,"The question asks about a company headquartered in Whitley, Covent...",Jaguar Land Rover,✔️ [1]
96,Who starred as an American attorney in Grey Gardens?,Ken Howard,Grey Gardens is a 1975 documentary film directed by Maysles Brothe...,No attorney,✔️ [0]
97,Was Princess Charlotte of Cambridge born before or after the repea...,after,The Royal Marriages Act 1772 was repealed in 2011. Princess Charlo...,after,✔️ [1]
98,Hermann Wilhelm Göring was a veteran fighter pilot in a war that e...,1918,Hermann Wilhelm Göring was a prominent German figure who served as...,1918,✔️ [1]


EvaluationResult(score=30.0, results=<list of 100 results>)

Generative AI was used to assist in building this notebook.